# 11 Composite | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: drzewiaste struktury danych
2. 🌿 Uczestnicy: Component / Leaf / Composite
3. 🔁 Rekurencja jako rdzen wzorca
4. 🌳 Implementacja: drzewo plikow
5. 🗺️ Kiedy uzywac Composite

## 1. 🔹 Problem: drzewiaste struktury danych

Composite (Kompozyt) to wzorzec strukturalny pozwalajacy
tworzyc drzewiaste struktury obiektow i traktowac
poszczegolne obiekty i kolekcje jednakowo.

Problem:
- Mamy proste obiekty (lisc) i zlozone (kontener)
- Kontener zawiera liscie i/lub inne kontenery
- Chcemy wykonywac operacje na calej strukturze
  bez rozrozniania typow

Przykłady z zycia:
- System plikow: plik i katalog maja operacje `size()`, `list()`
- Menu aplikacji: pozycja i podmenu maja `render()`
- Wyrazenia matematyczne: liczba i operacja maja `evaluate()`
- Organizacja firmy: pracownik i dzial maja `get_salary()`

> 💡 Composite pozwala pisac `root.get_size()` bez sprawdzania
> czy root to plik czy katalog - rekurencja zalatwia reszte.

In [ ]:
# Problem bez Composite: sprawdzamy typy
class File:
    def __init__(self, name: str, size: int): self.name = name; self.size = size

class Directory:
    def __init__(self, name: str): self.name = name; self.children = []

def get_total_size(item) -> int:
    # Musimy sprawdzac typ!
    if isinstance(item, File):
        return item.size
    elif isinstance(item, Directory):
        total = 0
        for child in item.children:
            total += get_total_size(child)  # rekurencja manualna
        return total
    raise TypeError(f'Unknown type: {type(item)}')

root = Directory('root')
root.children.append(File('a.py', 100))
sub = Directory('sub')
sub.children.append(File('b.py', 200))
root.children.append(sub)

print(f'Size: {get_total_size(root)}')
print('Problem: isinstance sprawdza typ -> narusza OCP')
print('Problem: dodanie nowego typu wymaga zmiany get_total_size()')

---

### 🐍 Cwiczenia - problem

1. Napisz `count_files(item)` korzystajac z `isinstance`.
   Zlicz tylko pliki (nie katalogi) w drzewie.
2. Ile miejsc kodu musisz zmienic gdy dodasz nowy typ `SymLink`?
   Policz wszystkie funkcje operujace na strukturze.
3. *(Trudniejsze)* Przepisz `get_total_size` bez `isinstance`
   uzywajac duck typing i atrybutu `children` jako wskazu na Composite.

In [ ]:
# Cwiczenie 1: count_files z isinstance
def count_files(item) -> int:
    ...

print(f'Files in root: {count_files(root)}')

In [ ]:
# Cwiczenie 2: analiza kosztu
functions_to_change = ['get_total_size', 'count_files']
print(f'Funkcji do zmiany przy dodaniu SymLink: {len(functions_to_change)}')
print('Z Composite: 0 - dodajemy nowa klase implementujaca interfejs')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: duck typing
def get_size_duck(item) -> int:
    # hint: sprawdz czy item ma atrybut 'children'
    # jesli tak -> Directory, jesli nie -> File
    ...

print(f'Duck size: {get_size_duck(root)}')

## 2. 🔹 Uczestnicy: Component / Leaf / Composite

Struktura wzorca Composite:

- **Component** (abstrakcja):
  - Deklaruje interfejs wspolny dla Leaf i Composite
  - Moze implementowac domyslne zachowanie
  - Opcjonalnie: metody add/remove (transparentnosc vs bezpieczenstwo)

- **Leaf** (lisc):
  - Nie ma dzieci - to koniec galezia
  - Implementuje wszystkie operacje Component
  - Nie implementuje add/remove

- **Composite** (wezel):
  - Przechowuje liste dzieci (Component)
  - Implementuje operacje przez rekurencje na dzieciach
  - Implementuje add/remove

Dwa podejscia do interfejsu:
- Transparency: add/remove w Component - wszystko jednakowo
- Safety: add/remove tylko w Composite - kompilator pilnuje

> 💡 Python czesciej uzywa safety approach - add() i remove()
> sa dostepne tylko na Composite (Directory), nie na Leaf (File).

In [ ]:
from abc import ABC, abstractmethod

# Component: wspolny interfejs
class FileSystemComponent(ABC):
    @abstractmethod
    def get_size(self) -> int: ...
    @abstractmethod
    def display(self, indent: int = 0) -> None: ...
    @property
    @abstractmethod
    def name(self) -> str: ...

# Leaf: plik
class File(FileSystemComponent):
    def __init__(self, name: str, size: int):
        self._name = name
        self._size = size

    @property
    def name(self) -> str: return self._name

    def get_size(self) -> int: return self._size

    def display(self, indent: int = 0) -> None:
        print(' ' * indent + f'- {self._name} ({self._size} B)')

# Composite: katalog
class Directory(FileSystemComponent):
    def __init__(self, name: str):
        self._name = name
        self._children: list[FileSystemComponent] = []

    @property
    def name(self) -> str: return self._name

    def add(self, component: FileSystemComponent) -> None:
        self._children.append(component)

    def remove(self, component: FileSystemComponent) -> None:
        self._children.remove(component)

    def get_size(self) -> int:
        return sum(child.get_size() for child in self._children)

    def display(self, indent: int = 0) -> None:
        print(' ' * indent + f'+ {self._name}/ ({self.get_size()} B)')
        for child in self._children:
            child.display(indent + 2)

# Klient traktuje wszystko jednakowo
root = Directory('root')
docs = Directory('docs')
docs.add(File('readme.md', 1024))
docs.add(File('spec.pdf', 51200))
src = Directory('src')
src.add(File('main.py', 2048))
src.add(File('utils.py', 4096))
root.add(docs)
root.add(src)
root.add(File('config.json', 512))

root.display()
print(f'Total: {root.get_size()} B')

# Klient nie wie czy ma File czy Directory!
def print_info(component: FileSystemComponent) -> None:
    print(f'{component.name}: {component.get_size()} B')

for item in [File('x.txt', 100), Directory('empty')]:
    print_info(item)  # ten sam interfejs!

---

### 🐍 Cwiczenia - Component / Leaf / Composite

1. Dodaj do `FileSystemComponent` metode `count_files() -> int`.
   Leaf zwraca 1, Directory zwraca sume dzieci.
2. Napisz hierarchie `Shape` (Component), `Circle` i `Square` (Leaf),
   `ShapeGroup` (Composite). Interfejs: `area() -> float`, `display()`.
3. *(Trudniejsze)* Dodaj do Composite metode `find(name: str)`
   zwracajaca component o podanej nazwie (przeszukiwanie DFS).

In [ ]:
# Cwiczenie 1: count_files
class FileSystemComponent2(ABC):
    @abstractmethod
    def get_size(self) -> int: ...
    @abstractmethod
    def count_files(self) -> int: ...
    @abstractmethod
    def display(self, indent: int = 0) -> None: ...

class File2(FileSystemComponent2):
    def __init__(self, name: str, size: int): self.name = name; self._size = size
    def get_size(self) -> int: return self._size
    def count_files(self) -> int: ...
    def display(self, indent: int = 0) -> None: print(' ' * indent + f'- {self.name}')

class Directory2(FileSystemComponent2):
    def __init__(self, name: str): self.name = name; self._children = []
    def add(self, c): self._children.append(c)
    def get_size(self) -> int: return sum(c.get_size() for c in self._children)
    def count_files(self) -> int: ...
    def display(self, indent: int = 0) -> None:
        print(' ' * indent + f'+ {self.name}/')
        for c in self._children: c.display(indent + 2)

d = Directory2('root')
d.add(File2('a.py', 100))
d.add(File2('b.py', 200))
sub = Directory2('sub')
sub.add(File2('c.py', 300))
d.add(sub)
print(f'Files: {d.count_files()}')  # 3

In [ ]:
# Cwiczenie 2: Shape hierarchy
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...
    @abstractmethod
    def display(self, indent: int = 0) -> None: ...

class Circle(Shape):
    def __init__(self, radius: float): self.radius = radius
    def area(self) -> float: ...
    def display(self, indent: int = 0) -> None: ...

class Square(Shape):
    def __init__(self, side: float): self.side = side
    def area(self) -> float: ...
    def display(self, indent: int = 0) -> None: ...

class ShapeGroup(Shape):
    def __init__(self, name: str): self.name = name; self._shapes = []
    def add(self, s: Shape) -> None: self._shapes.append(s)
    def area(self) -> float: ...
    def display(self, indent: int = 0) -> None: ...

group = ShapeGroup('scene')
group.add(Circle(5))
group.add(Square(4))
inner = ShapeGroup('inner')
inner.add(Circle(2))
group.add(inner)
group.display()
print(f'Total area: {group.area():.2f}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: find() DFS
class DirectoryWithFind(Directory):
    def find(self, name: str) -> 'FileSystemComponent | None':
        # hint: sprawdz siebie, potem rekurencyjnie dzieci
        ...

root2 = DirectoryWithFind('root')
src2 = DirectoryWithFind('src')
src2.add(File('main.py', 2048))
root2.add(src2)
root2.add(File('config.json', 512))

result = root2.find('main.py')
print(f'Found: {result.name if result else None}')
print(f'Not found: {root2.find("missing.txt")}')

## 3. 🔹 Rekurencja jako rdzen wzorca

Kluczem do Composite jest rekurencja w metodach Composite:

```python
def get_size(self) -> int:
    return sum(child.get_size() for child in self._children)
```

- `child.get_size()` wywoluje metode interfejsu
- Nie wiemy czy child to Leaf czy kolejny Composite
- Jesli Leaf: zwraca wartosc bezposrednio
- Jesli Composite: rekurencyjnie sumuje swoje dzieci

Obchodzenie drzewa (tree traversal):
- DFS (Depth-First): wglab - rekurencja naturalna
- BFS (Breadth-First): wszerz - kolejka (queue)
- Pre-order: rodzic przed dziecmi
- Post-order: dzieci przed rodzicem

Przypadki bazowe rekurencji:
- Plik (Leaf): `get_size()` zwraca `self._size`
- Pusty katalog: `sum([]) == 0`
- Brak dopasowania: `find()` zwraca `None`

In [ ]:
from abc import ABC, abstractmethod
from collections import deque

class TreeNode(ABC):
    @abstractmethod
    def value(self) -> int: ...
    @abstractmethod
    def accept_visitor(self, visitor) -> None: ...

class Leaf(TreeNode):
    def __init__(self, val: int): self._val = val
    def value(self) -> int: return self._val
    def accept_visitor(self, visitor) -> None: visitor(self)

class Node(TreeNode):
    def __init__(self, val: int):
        self._val = val
        self._children: list[TreeNode] = []
    def add(self, child: TreeNode) -> 'Node':
        self._children.append(child); return self
    def value(self) -> int: return self._val
    def accept_visitor(self, visitor) -> None:
        visitor(self)
        for child in self._children:
            child.accept_visitor(visitor)

    # DFS pre-order (rekurencja)
    def dfs_preorder(self) -> list:
        result = [self._val]
        for child in self._children:
            if isinstance(child, Node):
                result.extend(child.dfs_preorder())
            else:
                result.append(child.value())
        return result

    # BFS (kolejka)
    def bfs(self) -> list:
        result = []
        queue = deque([self])
        while queue:
            node = queue.popleft()
            result.append(node.value())
            if isinstance(node, Node):
                for child in node._children:
                    queue.append(child)
        return result

    # Suma wszystkich wartosci (rekurencja)
    def total(self) -> int:
        s = self._val
        for child in self._children:
            s += child.value() if isinstance(child, Leaf) else child.total()
        return s

# Budujemy drzewo:
#      1
#    /   \
#   2     3
#  / \
# 4   5
tree = Node(1).add(Node(2).add(Leaf(4)).add(Leaf(5))).add(Node(3))

print('DFS pre-order:', tree.dfs_preorder())
print('BFS:', tree.bfs())
print('Total:', tree.total())

---

### 🐍 Cwiczenia - rekurencja

1. Zaimplementuj `depth() -> int` dla drzewa plikow:
   File zwraca 0, Directory zwraca 1 + max glebokosci dzieci.
2. Zaimplementuj `flatten() -> list` zwracajacy wszystkie pliki
   w kolejnosci DFS (pre-order).
3. *(Trudniejsze)* Napisz `map_tree(root, func)` aplikujacy func
   do wartosci kazdego Leaf i zwracajacy nowe drzewo.

In [ ]:
# Cwiczenie 1: depth()
class FileDepth(ABC):
    @abstractmethod
    def depth(self) -> int: ...
    @abstractmethod
    def get_size(self) -> int: ...

class FileD(FileDepth):
    def __init__(self, name: str, size: int): self.name = name; self._size = size
    def depth(self) -> int: return 0
    def get_size(self) -> int: return self._size

class DirD(FileDepth):
    def __init__(self, name: str): self.name = name; self._ch = []
    def add(self, c): self._ch.append(c); return self
    def depth(self) -> int: return 1 + (max(c.depth() for c in self._ch) if self._ch else 0)
    def get_size(self) -> int: return sum(c.get_size() for c in self._ch)

root3 = DirD('root').add(DirD('a').add(DirD('b').add(FileD('x', 10))))
print(f'Depth: {root3.depth()}')  # 3

In [ ]:
# Cwiczenie 2: flatten() -> lista plikow
def flatten_files(component) -> list:
    # hint: jesli ma _children -> Composite, inaczej -> Leaf
    ...

root4 = Directory('root')
root4.add(File('a.py', 10))
sub4 = Directory('sub')
sub4.add(File('b.py', 20))
sub4.add(File('c.py', 30))
root4.add(sub4)
print('Files:', [f.name for f in flatten_files(root4)])

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: map_tree
from copy import deepcopy

def map_tree(node: TreeNode, func) -> TreeNode:
    # hint: dla Leaf: Leaf(func(val)), dla Node: Node z zmapowanymi dziecmi
    ...

tree2 = Node(1).add(Leaf(4)).add(Node(2).add(Leaf(5)))
print('Original:', tree2.dfs_preorder())
doubled = map_tree(tree2, lambda x: x * 2)
print('Doubled:', doubled.dfs_preorder())

## 4. 🔹 Implementacja: drzewo plikow

Pelna implementacja systemu plikow z rozbudowanym API:
- `get_size()` - rozmiar w bajtach
- `display(indent)` - wyswietlenie drzewa
- `find(name)` - wyszukiwanie pliku/katalogu
- `count_files()` - liczba plikow
- `__iter__` - iteracja po wszystkich komponentach

Wzorzec budowania drzewa:
- Metoda `add()` zwraca `self` dla fluent interface
- Gleboka kopia z `copy.deepcopy`
- Serializacja do slownika dla JSON

Operacje na drzewach:
- Merge: polaczenie dwoch katalogów
- Move: przeniesienie pliku/katalogu
- Clone: glebokie kopiowanie poddrzewa

In [ ]:
from abc import ABC, abstractmethod
from typing import Iterator

class FSComponent(ABC):
    @abstractmethod
    def get_size(self) -> int: ...
    @abstractmethod
    def display(self, indent: int = 0) -> None: ...
    @abstractmethod
    def find(self, name: str) -> 'FSComponent | None': ...
    @abstractmethod
    def count_files(self) -> int: ...
    @abstractmethod
    def to_dict(self) -> dict: ...

class FSFile(FSComponent):
    def __init__(self, name: str, size: int, ext: str = ''):
        self.name = name
        self._size = size
        self.ext = ext

    def get_size(self) -> int: return self._size

    def display(self, indent: int = 0) -> None:
        print(' ' * indent + f'- {self.name} ({self._size:,} B)')

    def find(self, name: str) -> 'FSComponent | None':
        return self if self.name == name else None

    def count_files(self) -> int: return 1

    def to_dict(self) -> dict:
        return {'type': 'file', 'name': self.name, 'size': self._size}

class FSDirectory(FSComponent):
    def __init__(self, name: str):
        self.name = name
        self._children: list[FSComponent] = []

    def add(self, component: FSComponent) -> 'FSDirectory':
        self._children.append(component)
        return self

    def get_size(self) -> int:
        return sum(c.get_size() for c in self._children)

    def display(self, indent: int = 0) -> None:
        size_kb = self.get_size() / 1024
        print(' ' * indent + f'+ {self.name}/ ({size_kb:.1f} KB)')
        for child in self._children:
            child.display(indent + 2)

    def find(self, name: str) -> 'FSComponent | None':
        if self.name == name:
            return self
        for child in self._children:
            result = child.find(name)
            if result is not None:
                return result
        return None

    def count_files(self) -> int:
        return sum(c.count_files() for c in self._children)

    def to_dict(self) -> dict:
        return {
            'type': 'directory',
            'name': self.name,
            'children': [c.to_dict() for c in self._children],
        }

# Budowanie struktury
project = (FSDirectory('my_project')
    .add(FSDirectory('src')
        .add(FSFile('main.py', 2048))
        .add(FSFile('utils.py', 4096))
        .add(FSDirectory('models')
            .add(FSFile('user.py', 1024))
            .add(FSFile('product.py', 1536))))
    .add(FSDirectory('tests')
        .add(FSFile('test_main.py', 3072))
        .add(FSFile('test_utils.py', 2048)))
    .add(FSFile('README.md', 512))
    .add(FSFile('requirements.txt', 256)))

project.display()
print(f'\nTotal: {project.get_size():,} B')
print(f'Files: {project.count_files()}')
print(f'Find: {project.find("user.py").name}')

---

### 🐍 Cwiczenia - system plikow

1. Dodaj metode `find_by_extension(ext: str) -> list` do `FSDirectory`
   zwracajaca wszystkie pliki z danym rozszerzeniem.
2. Zaimplementuj `to_dict()` dla obu klas i sprawdz ze mozna
   serializowac do JSON (`json.dumps`).
3. *(Trudniejsze)* Napisz funkcje `diff(dir1, dir2) -> dict`
   zwracajaca `{'only_in_1': list, 'only_in_2': list, 'common': list}`.

In [ ]:
# Cwiczenie 1: find_by_extension
class FSDirectoryExt(FSDirectory):
    def find_by_extension(self, ext: str) -> list:
        # hint: rekurencja, sprawdz czy name konczy sie na '.ext'
        results = []
        for child in self._children:
            if isinstance(child, FSFile) and child.name.endswith(f'.{ext}'):
                results.append(child)
            elif isinstance(child, FSDirectoryExt):
                results.extend(child.find_by_extension(ext))
        return results

# Zbuduj strukture z FSDirectoryExt
proj2 = FSDirectoryExt('proj')
proj2.add(FSFile('main.py', 100))
proj2.add(FSFile('style.css', 200))
sub = FSDirectoryExt('sub')
sub.add(FSFile('utils.py', 150))
proj2.add(sub)
py_files = proj2.find_by_extension('py')
print('Python files:', [f.name for f in py_files])

In [ ]:
# Cwiczenie 2: serializacja JSON
import json

small = FSDirectory('proj').add(FSFile('main.py', 100)).add(FSFile('config.json', 50))
data = small.to_dict()
print(json.dumps(data, indent=2))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: diff
def get_all_names(component: FSComponent, prefix: str = '') -> set:
    if isinstance(component, FSFile):
        return {prefix + component.name}
    result = set()
    for child in component._children:
        result |= get_all_names(child, prefix + component.name + '/')
    return result

def diff(dir1: FSDirectory, dir2: FSDirectory) -> dict:
    names1 = get_all_names(dir1)
    names2 = get_all_names(dir2)
    return {
        'only_in_1': sorted(names1 - names2),
        'only_in_2': sorted(names2 - names1),
        'common': sorted(names1 & names2),
    }

d1 = FSDirectory('d1').add(FSFile('a.py', 100)).add(FSFile('b.py', 200))
d2 = FSDirectory('d2').add(FSFile('b.py', 200)).add(FSFile('c.py', 300))
result = diff(d1, d2)
print('Only in d1:', result['only_in_1'])
print('Only in d2:', result['only_in_2'])
print('Common:', result['common'])

## 5. 🔹 Kiedy uzywac Composite

Composite stosuj gdy:
1. Dane maja naturalna hierarchie drzewiasta
2. Chcesz traktowac poszczegolne obiekty i grupy jednakowo
3. Kod klienta nie powinien rozroznic Leaf od Composite

Nie stosuj gdy:
- Struktura nie jest rekurencyjna (nie hierarchiczna)
- Rozroznienie Leaf/Composite jest potrzebne klientowi
- Wydajnosc krytyczna (rekurencja ma koszt stacku)

Zwiazek z innymi wzorcami:
- **Iterator**: obchodzenie drzewa Composite
- **Visitor**: operacje na drzewie bez modyfikacji klas
- **Decorator**: rowniez rekurencyjny, ale nie drzewo
- **Builder**: budowanie zlozonych struktur Composite
- **Flyweight**: optymalizacja lisci w duzych drzewach

Przykladowe struktury Composite w praktyce:
- DOM (Document Object Model) w przegladarkach
- GUI komponenty (Panel zawiera widgety i inne panele)
- YAML/JSON zagniezdzone struktury
- Zapytania SQL z podzapytaniami
- Sceny graficzne (SceneNode)

In [ ]:
# Wyrazenia SQL jako Composite
class SQLExpression(ABC):
    @abstractmethod
    def to_sql(self) -> str: ...

class Column(SQLExpression):
    def __init__(self, name: str): self.name = name
    def to_sql(self) -> str: return self.name

class Literal(SQLExpression):
    def __init__(self, value): self.value = value
    def to_sql(self) -> str:
        return f"'{self.value}'" if isinstance(self.value, str) else str(self.value)

class Condition(SQLExpression):
    def __init__(self, left: SQLExpression, op: str, right: SQLExpression):
        self.left = left; self.op = op; self.right = right
    def to_sql(self) -> str:
        return f'({self.left.to_sql()} {self.op} {self.right.to_sql()})'

class AndCondition(SQLExpression):
    def __init__(self, *conditions: SQLExpression):
        self.conditions = conditions
    def to_sql(self) -> str:
        return ' AND '.join(c.to_sql() for c in self.conditions)

class OrCondition(SQLExpression):
    def __init__(self, *conditions: SQLExpression):
        self.conditions = conditions
    def to_sql(self) -> str:
        return '(' + ' OR '.join(c.to_sql() for c in self.conditions) + ')'

# WHERE (age > 18 AND city = 'Warsaw') OR (role = 'admin')
where = OrCondition(
    AndCondition(
        Condition(Column('age'), '>', Literal(18)),
        Condition(Column('city'), '=', Literal('Warsaw'))
    ),
    Condition(Column('role'), '=', Literal('admin'))
)
print('WHERE', where.to_sql())

---

### 🐍 Cwiczenia - zastosowania

1. Napisz `HTMLElement` (Composite) generujacy HTML.
   `Div` i `Span` to Composite, `Text` to Leaf.
2. Napisz `NumberExpr` z `Number`, `Add`, `Sub`, `Mul`, `Div`.
   Oblicz `(10 + 5) * (8 - 3) / 2`.
3. *(Trudniejsze)* Napisz Composite reprezentujacy menu
   aplikacji. Metody: `render()`, `find_item(label)`,
   `count_items()`. Menu zawiera item-y i pod-menu.

In [ ]:
# Cwiczenie 1: HTML Composite
class HTMLComponent(ABC):
    @abstractmethod
    def render(self) -> str: ...

class Text(HTMLComponent):
    def __init__(self, content: str): self.content = content
    def render(self) -> str: return self.content

class HTMLElement(HTMLComponent):
    def __init__(self, tag: str, **attrs):
        self.tag = tag
        self.attrs = attrs
        self._children: list[HTMLComponent] = []
    def add(self, child: HTMLComponent) -> 'HTMLElement':
        self._children.append(child); return self
    def render(self) -> str:
        attr_str = ''.join(f' {k}="{v}"' for k, v in self.attrs.items())
        inner = ''.join(c.render() for c in self._children)
        return f'<{self.tag}{attr_str}>{inner}</{self.tag}>'

page = (HTMLElement('div', id='app')
    .add(HTMLElement('h1').add(Text('Composite Pattern')))
    .add(HTMLElement('p', class_='info').add(Text('Tree structure')))
    .add(HTMLElement('ul')
        .add(HTMLElement('li').add(Text('Leaf')))
        .add(HTMLElement('li').add(Text('Composite')))))
print(page.render())

In [ ]:
# Cwiczenie 2: wyrazenia numeryczne
class NumberExpr(ABC):
    @abstractmethod
    def evaluate(self) -> float: ...

class Num(NumberExpr):
    def __init__(self, v: float): self.v = v
    def evaluate(self) -> float: return self.v

class Add(NumberExpr):
    def __init__(self, a, b): self.a = a; self.b = b
    def evaluate(self) -> float: return self.a.evaluate() + self.b.evaluate()

class Sub(NumberExpr):
    def __init__(self, a, b): self.a = a; self.b = b
    def evaluate(self) -> float: return self.a.evaluate() - self.b.evaluate()

class Mul(NumberExpr):
    def __init__(self, a, b): self.a = a; self.b = b
    def evaluate(self) -> float: return self.a.evaluate() * self.b.evaluate()

class Div(NumberExpr):
    def __init__(self, a, b): self.a = a; self.b = b
    def evaluate(self) -> float: return self.a.evaluate() / self.b.evaluate()

# (10 + 5) * (8 - 3) / 2
expr2 = Div(Mul(Add(Num(10), Num(5)), Sub(Num(8), Num(3))), Num(2))
print(f'(10+5)*(8-3)/2 = {expr2.evaluate()}')  # 37.5

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: menu aplikacji
class MenuComponent2(ABC):
    @abstractmethod
    def render(self, indent: int = 0) -> None: ...
    @abstractmethod
    def find_item(self, label: str) -> 'MenuComponent2 | None': ...
    @abstractmethod
    def count_items(self) -> int: ...

class MenuItem2(MenuComponent2):
    def __init__(self, label: str, shortcut: str = ''):
        self.label = label; self.shortcut = shortcut
    def render(self, indent: int = 0) -> None:
        sc = f'  [{self.shortcut}]' if self.shortcut else ''
        print(' ' * indent + f'- {self.label}{sc}')
    def find_item(self, label: str): return self if self.label == label else None
    def count_items(self) -> int: return 1

class SubMenu(MenuComponent2):
    def __init__(self, label: str):
        self.label = label; self._items = []
    def add(self, item): self._items.append(item); return self
    def render(self, indent: int = 0) -> None:
        print(' ' * indent + f'[+] {self.label}')
        for item in self._items: item.render(indent + 2)
    def find_item(self, label: str):
        if self.label == label: return self
        for item in self._items:
            r = item.find_item(label)
            if r: return r
        return None
    def count_items(self) -> int:
        return sum(i.count_items() for i in self._items)

menu = (SubMenu('File')
    .add(MenuItem2('New', 'Ctrl+N'))
    .add(MenuItem2('Open', 'Ctrl+O'))
    .add(SubMenu('Recent')
        .add(MenuItem2('file1.py'))
        .add(MenuItem2('file2.py')))
    .add(MenuItem2('Exit', 'Alt+F4')))

menu.render()
print(f'Total items: {menu.count_items()}')
found = menu.find_item('file1.py')
print(f'Found: {found.label if found else None}')